In [61]:
from langchain_core.tools import tool

@tool 
async def division(a:int,b:int)->int:
    """Returns the division of a by b"""
    return a/b

In [62]:
print(division.name)
print(division.description)
print(division.args)

division
Returns the division of a by b
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [63]:
from typing import Annotated,List
@tool 
def multiply_by_max(
    a:Annotated[int ," A value"],
    b:Annotated[List[int] ," B list value"]
)->int:
   """Multiplies a with maximum  b """
   return a*max(b)

In [64]:
print(multiply_by_max.args)

{'a': {'description': ' A value', 'title': 'A', 'type': 'integer'}, 'b': {'description': ' B list value', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


In [65]:
multiply_by_max.invoke({"a": 2, "b": [1,2,3]}) 

6

In [66]:
await division.ainvoke({"a": 10, "b": 2})

5.0

In [67]:
from langchain_core.tools import StructuredTool

def division(a:int,b:int)->int:
    """Returns the division of a by b"""
    return a/b

async def adivision(a:int,b:int)->int:
    """Returns the division of a by b"""
    return a/b
calculator=StructuredTool.from_function(func=division,coroutine=adivision)

print(calculator.invoke({"a": 10, "b": 2}))
print(await calculator.ainvoke({"a": 10, "b": 2}))

5.0
5.0


In [68]:
##In built tools
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=500)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.invoke({"query": "Is gurgaon near Delhi"}))


Page: Gurgaon
Summary: Gurgaon (Hindi: [ɡʊɽɡãːw]), officially named Gurugram ([ɡʊɾʊɡɾaːm]), is a satellite city of Delhi and administrative headquarters of Gurgaon district, located in the northern Indian state of Haryana. It is situated near the Delhi–Haryana border, about 30 kilometres (19 mi) southwest of the national capital New Delhi and 268 km (167 mi) south of Chandigarh, the state capital. It is one of the major satellite cities of Delhi and is part of the National Capital Region of Indi


In [69]:
from langchain_community.utilities import ArxivAPIWrapper
arxiv = ArxivAPIWrapper()
docs = arxiv.run("1605.08386")
docs

'Published: 2016-05-26\nTitle: Heat-bath random walks with Markov bases\nAuthors: Caprice Stanley, Tobias Windisch\nSummary: Graphs on lattice points are studied whose edges come from a finite set of allowed moves of arbitrary length. We show that the diameter of these graphs on fibers of a fixed integer matrix can be bounded from above by a constant. We then study the mixing behaviour of heat-bath random walks on these graphs. We also state explicit conditions on the set of moves so that the heat-bath random walk, a generalization of the Glauber dynamics, is an expander in fixed dimension.'

In [70]:
###Call tool with the llm model
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=500)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(wiki_tool.invoke({"query": "langchain"}))


Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in v


In [71]:
from langchain_core.tools import tool
@tool
def add(a:int,b:int)->int:
    """Returns the sum of a and b"""
    return a+b  

@tool
def subtract(a:int,b:int)->int:     
    """Returns the difference of a and b"""
    return a-b

In [72]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun
arxiv_wrapper = ArxivAPIWrapper()
arxiv_tool=arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)


In [73]:
tools=[wiki_tool,add,subtract,arxiv_tool]

In [74]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\JAYA AGRWAL\\Desktop\\genai codes\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=5, lang='en', load_all_available_meta=False, doc_content_chars_max=500)),
 StructuredTool(name='add', description='Returns the sum of a and b', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x0000010F057DC540>),
 StructuredTool(name='subtract', description='Returns the difference of a and b', args_schema=<class 'langchain_core.utils.pydantic.subtract'>, func=<function subtract at 0x0000010F057DC4A0>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000))]

In [75]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("qwen/qwen3-32b",model_provider='groq')
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000010F0577D940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000010F0577CFC0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [76]:
llm_tool=llm.bind_tools(tools)

In [77]:
from langchain_core.messages import HumanMessage
query="What is langchain? Also what are the recent papers on langchain in arxiv? And what is 10+20 and 100-50?"
messages=[HumanMessage(content=query)]
response=llm_tool.invoke(messages)
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, let\'s tackle the user\'s query step by step. First, they want to know what LangChain is. Since I need to provide an accurate definition, I should use the Wikipedia function to look it up. The user also asked for recent papers on LangChain in arXiv, so I\'ll need to use the arXiv function for that part. Lastly, there are two arithmetic questions: 10+20 and 100-50. Those are straightforward, so I can use the add and subtract functions respectively.\n\nStarting with the first part, I\'ll call Wikipedia with the query "LangChain" to get a general overview. Then, for the recent papers, I\'ll search arXiv with the same term. The math problems are simple, so adding 10 and 20 should give 30, and subtracting 50 from 100 should give 50. I need to make sure each function call is correctly formatted and that the parameters are properly structured as JSON objects within the tool_call tags. Let me put it all together now.\n', 'tool_calls': [

In [78]:
response.tool_calls

[{'name': 'wikipedia',
  'args': {'query': 'LangChain'},
  'id': 'rbqh0snyy',
  'type': 'tool_call'},
 {'name': 'arxiv',
  'args': {'query': 'LangChain'},
  'id': 'vh14yyjkc',
  'type': 'tool_call'},
 {'name': 'add',
  'args': {'a': 10, 'b': 20},
  'id': '6j5n5nvh8',
  'type': 'tool_call'},
 {'name': 'subtract',
  'args': {'a': 100, 'b': 50},
  'id': 'r2n47r4zv',
  'type': 'tool_call'}]

In [79]:
for tool_call in response.tool_calls:
    selected_tool={'add':add,'subtract':subtract,'wikipedia':wiki_tool,'arxiv':arxiv_tool}[tool_call["name"].lower()]
    tool_response=selected_tool.invoke(tool_call)
    messages.append(tool_response)
messages

[HumanMessage(content='What is langchain? Also what are the recent papers on langchain in arxiv? And what is 10+20 and 100-50?', additional_kwargs={}, response_metadata={}),
 ToolMessage(content="Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.\n\n\n\nPage: Model Context Protocol\nSummary: The Model Context Protocol (MCP) is an open standard and open-source framework introduced by Anthropic in Novemb", name='wikipedia', tool_call_id='rbqh0snyy'),
 ToolMessage(content="Published: 2023-12-05\nTitle: Breast Ultrasound Report Generation using LangChain\nAuthors: Jaeyoung Huh, Hyun Jeong Park, Jong Chul Ye\nSummary: Breast ultrasound (BUS) is a critical diagnostic tool in the field of breast imagi

In [81]:
ans=llm_tool.invoke(messages)
ans.content

'1. **What is LangChain?**  \nLangChain is a software framework designed to integrate large language models (LLMs) into applications. It supports use cases like document analysis, chatbots, and code analysis by providing tools for managing model interactions, retrieval-augmented generation (RAG), and workflow automation.  \n\n2. **Recent arXiv Papers on LangChain:**  \n   - **Breast Ultrasound Report Generation using LangChain** (2023-12-05)  \n     *Authors:* Jaeyoung Huh, Hyun Jeong Park, Jong Chul Ye  \n     *Summary:* Combines LangChain with LLMs to automate standardized medical reporting for breast ultrasound, improving efficiency and consistency.  \n\n   - **Poisoned LangChain: Jailbreak LLMs by LangChain** (2024-06-26)  \n     *Authors:* Ziqiu Wang, Jun Liu, Shengkai Zhang, Yang Yang  \n     *Summary:* Introduces "Poisoned-LangChain," a method to exploit LangChain\'s external knowledge base for indirect jailbreak attacks on LLMs, bypassing safety mechanisms.  \n\n   - **Automati